# SaaS Implementation & Integration Health

## tl;dr

The synthetic portfolio has **5 active implementations**, **5 red/amber accounts**, an on-time completed go-live rate of **45.5%**, and median time to first value of **51 days**. The first intervention account is **ACCT-058**.


## Context & Methods

Decision: which implementations risk missing committed go-live and what should happen next? Risk is rule-based, not an opaque score.

### Key Assumptions

- Completed and active projects use different eligibility rules.
- Time to first value requires sync success, data acceptance, user adoption and a completed workflow.
- Later template cohorts are observational synthetic comparisons, not causal evidence.


In [1]:
from pathlib import Path
import csv, json
PROJECT_DIR = Path.cwd()
def read_csv(name):
    with (PROJECT_DIR / name).open(encoding='utf-8') as handle:
        return list(csv.DictReader(handle))
summary = json.loads((PROJECT_DIR / 'data/curated/summary.json').read_text())
risk = read_csv('data/curated/risk_queue.csv')
print(json.dumps(summary, indent=2))
print(f"active accounts in intervention view={len(risk)}")


{
  "active_implementations": 5,
  "on_time_go_live_rate": 0.454545,
  "median_ttfv_days": 51.0,
  "at_risk_active_accounts": 5,
  "integration_success_rate": 0.980418
}
active accounts in intervention view=5


## Data

Account, connector, milestone, mapping, UAT, sync-run, blocker and usage tables form a complete synthetic implementation lifecycle.


In [2]:
runs = read_csv('data/raw/integration_runs.csv')
received = sum(int(row['records_received']) for row in runs)
accepted = sum(int(row['records_accepted']) for row in runs)
success = sum(row['status'] == 'SUCCESS' for row in runs)
print(f"runs={len(runs):,} | success={success/len(runs):.2%} | data acceptance={accepted/received:.2%}")


runs=9,192 | success=98.04% | data acceptance=97.93%


## Results

The intervention queue ranks active accounts by explicit red/amber rules and committed date. Each row carries the reason and next action.


In [3]:
for row in risk:
    print(f"{row['account_id']} | {row['risk_status']:<5} | days={row['days_to_committed_go_live']:>3} | mapping={float(row['mapping_coverage']):.0%} | UAT={float(row['uat_pass_rate']):.0%} | {row['next_action']}")


ACCT-058 | Red   | days= -8 | mapping=67% | UAT=100% | Run daily recovery plan with decision owner
ACCT-059 | Red   | days= -2 | mapping=67% | UAT=62% | Run daily recovery plan with decision owner
ACCT-057 | Red   | days=  2 | mapping=67% | UAT=50% | Run daily recovery plan with decision owner
ACCT-055 | Amber | days= 16 | mapping=100% | UAT=75% | Resolve blocker and re-baseline remaining milestones
ACCT-060 | Amber | days= 40 | mapping=0% | UAT=0% | Resolve blocker and re-baseline remaining milestones


## Takeaways

1. Act first on near-term red accounts with incomplete mapping or UAT.
2. Connector reliability and data acceptance must be reviewed together; a successful HTTP response does not prove usable data.
3. Treat cohort improvements as a process signal that needs controlled follow-up, not proof of causality.
